# Phase 2B Inference — Force & Speed Regression

Sliding window inference on a full video to predict peak force (N) and peak speed (m/s) for each detected punch.

**Inputs:**
- Full video `_pose_norm.npy` (from `main.py`)
- Trained TCN regressor checkpoint
- Subject's body weight (kg)
- Phase 2A TCN classifier (optional, to identify punch types alongside regression)

**Outputs:**
- Per-punch predictions: class, peak_force, peak_speed
- Annotated MP4 with overlaid predictions

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

## Configuration

Paths, model selection, subject body weight, and sliding window parameters.

In [ ]:
PROJECT_ROOT = Path("../../")

METADATA_DIR = PROJECT_ROOT / "data" / "metadata" / "with_hardware"
PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed" / "with_hardware"
LABELS_CSV = METADATA_DIR / "labels.csv"
SUBJECTS_CSV = METADATA_DIR / "subjects.csv"

# Model paths
REGRESSION_MODELS_DIR = PROJECT_ROOT / "models" / "tcn_regression"
CLASSIFICATION_MODELS_DIR = PROJECT_ROOT / "models" / "tcn"

# Model parameters (must match training)
T = 48
N_JOINTS = 9
N_CHANNELS = 3
N_FEATURES = N_JOINTS * N_CHANNELS
N_CLASSES = 5
CLASS_TO_IDX = {"cross": 0, "hook": 1, "jab": 2, "uppercut": 3, "no_punch": 4}
IDX_TO_CLASS = {v: k for k, v in CLASS_TO_IDX.items()}
PUNCH_CLASSES = {"cross", "hook", "jab", "uppercut"}

# Sliding window parameters (matched to Phase 2A)
WINDOW_STEP = 2
CONFIDENCE_THRESHOLD = 0.7
MIN_PUNCH_LENGTH = 4
MAX_GAP_TO_MERGE = 4

# List available regression models
available_reg_models = sorted(REGRESSION_MODELS_DIR.glob("**/best.pt"))
print(f"Available regression models in {REGRESSION_MODELS_DIR}:")
for m in available_reg_models:
    print(f"  {m.relative_to(REGRESSION_MODELS_DIR)}")

# Pick the most recent regression model by default
REGRESSION_MODEL_PATH = available_reg_models[-1] if available_reg_models else None
print(f"\nUsing regression model: {REGRESSION_MODEL_PATH}")

# Classification model (Phase 2A) — for punch type identification
available_cls_models = sorted(CLASSIFICATION_MODELS_DIR.glob("*_best.pt"))
print(f"\nAvailable classification models in {CLASSIFICATION_MODELS_DIR}:")
for m in available_cls_models:
    print(f"  {m.name}")
CLASSIFICATION_MODEL_PATH = available_cls_models[-1] if available_cls_models else None
print(f"\nUsing classification model: {CLASSIFICATION_MODEL_PATH}")

## Load Target Normalization Stats

The regression model was trained on normalized targets. We need the same
mean/std to un-normalize predictions back to N and m/s.

In [ ]:
# Recompute target normalization from the training labels
# (matches what the training notebook did per-fold)
labels_df = pd.read_csv(LABELS_CSV)

# For the demo, use stats from ALL training data
# In production with CV models, you'd load fold-specific stats
target_mean = labels_df[["peak_force_N", "peak_speed_mps"]].mean().values
target_std = labels_df[["peak_force_N", "peak_speed_mps"]].std().values

print(f"Target normalization:")
print(f"  Force: mean={target_mean[0]:.1f} N, std={target_std[0]:.1f} N")
print(f"  Speed: mean={target_mean[1]:.2f} m/s, std={target_std[1]:.2f} m/s")

target_mean_tensor = torch.tensor(target_mean, dtype=torch.float32).to(DEVICE)
target_std_tensor = torch.tensor(target_std, dtype=torch.float32).to(DEVICE)

## Model Architectures

Re-create both architectures (regression + classification) to load weights.

In [ ]:
class TCNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=5, dilation=1, dropout=0.2):
        super().__init__()
        padding = (kernel_size - 1) * dilation // 2
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size,
                               padding=padding, dilation=dilation)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size,
                               padding=padding, dilation=dilation)
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.dropout = nn.Dropout(dropout)
        if in_channels != out_channels:
            self.residual = nn.Conv1d(in_channels, out_channels, kernel_size=1)
        else:
            self.residual = nn.Identity()
        self.relu = nn.ReLU(inplace=True)
    
    def forward(self, x):
        res = self.residual(x)
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.dropout(x)
        x = self.bn2(self.conv2(x))
        x = self.relu(x + res)
        x = self.dropout(x)
        return x


class TCNRegressor(nn.Module):
    """Phase 2B: predicts peak force and peak speed."""
    def __init__(self, in_channels=N_FEATURES, channels=[64, 128, 256],
                 kernel_size=5, dropout=0.2, n_outputs=2):
        super().__init__()
        self.data_bn = nn.BatchNorm1d(in_channels)
        layers = []
        prev_channels = in_channels
        for i, ch in enumerate(channels):
            dilation = 2 ** i
            layers.append(TCNBlock(prev_channels, ch, kernel_size, dilation, dropout))
            prev_channels = ch
        self.tcn = nn.Sequential(*layers)
        self.fc = nn.Sequential(
            nn.Linear(prev_channels + 1, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(64, n_outputs),
        )
    
    def forward(self, x, body_weight):
        x = self.data_bn(x)
        x = self.tcn(x)
        x = x.mean(dim=2)
        body_weight = body_weight.unsqueeze(1)
        x = torch.cat([x, body_weight], dim=1)
        x = self.fc(x)
        return x


class TCNClassifier(nn.Module):
    """Phase 2A: predicts punch class."""
    def __init__(self, in_channels=N_FEATURES, n_classes=N_CLASSES,
                 channels=[64, 128, 256], kernel_size=5, dropout=0.2):
        super().__init__()
        self.data_bn = nn.BatchNorm1d(in_channels)
        layers = []
        prev_channels = in_channels
        for i, ch in enumerate(channels):
            dilation = 2 ** i
            layers.append(TCNBlock(prev_channels, ch, kernel_size, dilation, dropout))
            prev_channels = ch
        self.tcn = nn.Sequential(*layers)
        self.fc = nn.Linear(prev_channels, n_classes)
    
    def forward(self, x):
        x = self.data_bn(x)
        x = self.tcn(x)
        x = x.mean(dim=2)
        x = self.fc(x)
        return x


# Load both models
reg_model = TCNRegressor().to(DEVICE)
reg_model.load_state_dict(torch.load(REGRESSION_MODEL_PATH, map_location=DEVICE))
reg_model.eval()
print(f"Loaded regression model: {REGRESSION_MODEL_PATH.name}")
print(f"  Parameters: {sum(p.numel() for p in reg_model.parameters()):,}")

cls_model = TCNClassifier().to(DEVICE)
cls_model.load_state_dict(torch.load(CLASSIFICATION_MODEL_PATH, map_location=DEVICE))
cls_model.eval()
print(f"\nLoaded classification model: {CLASSIFICATION_MODEL_PATH.name}")
print(f"  Parameters: {sum(p.numel() for p in cls_model.parameters()):,}")

## Pick Test Video and Subject

The body weight is required for the regression model.

In [ ]:
# Discover available processed videos
available_videos = []
for subject_dir in sorted(PROCESSED_ROOT.iterdir()):
    if subject_dir.is_dir():
        for npy_path in subject_dir.glob("*_pose_norm.npy"):
            available_videos.append({
                "subject_id": subject_dir.name,
                "video_stem": npy_path.stem.replace("_pose_norm", ""),
                "pose_path": npy_path,
            })

print(f"Found {len(available_videos)} processed videos")
print("\nFirst 10:")
for v in available_videos[:10]:
    print(f"  {v['subject_id']} / {v['video_stem']}")

# Load subject body weights
subjects_df = pd.read_csv(SUBJECTS_CSV)

# Single test video override
USE_SINGLE_TEST_VIDEO = True
TEST_VIDEO_INDEX = 0  # if not single, picks by this index

if USE_SINGLE_TEST_VIDEO:
    TEST_VIDEO_NAME = "test_video"  # change to your video stem
    TEST_SUBJECT_ID = "subject01"   # for body weight lookup
    TEST_FOLDER = "test"            # subfolder under processed/with_hardware/
    
    test_video_path = PROCESSED_ROOT / TEST_FOLDER / f"{TEST_VIDEO_NAME}_pose_norm.npy"
    if not test_video_path.exists():
        # Fall back to processed/no_hardware/test
        test_video_path = PROJECT_ROOT / "data" / "processed" / "no_hardware" / "test" / f"{TEST_VIDEO_NAME}_pose_norm.npy"
    
    if not test_video_path.exists():
        raise FileNotFoundError(f"Test video pose file not found: {test_video_path}")
    
    test_video = {
        "subject_id": TEST_SUBJECT_ID,
        "video_stem": TEST_VIDEO_NAME,
        "pose_path": test_video_path,
    }
else:
    test_video = available_videos[TEST_VIDEO_INDEX]
    TEST_SUBJECT_ID = test_video["subject_id"]

# Look up body weight for this subject
subject_row = subjects_df[subjects_df["subject_id"] == TEST_SUBJECT_ID]
if len(subject_row) == 0:
    print(f"\n⚠️  Subject '{TEST_SUBJECT_ID}' not in subjects.csv")
    body_weight_kg = float(input("Enter body weight in kg: "))
else:
    body_weight_kg = float(subject_row["body_weight_kg"].iloc[0])

print(f"\nUsing test video: {test_video['pose_path']}")
print(f"Subject body weight: {body_weight_kg} kg")

## Sliding Window Inference Function

Slide a window of size T across the full pose array. For each window:
1. Run the classifier → punch type + confidence
2. If a punch is detected, run the regressor → force and speed predictions

In [ ]:
def sliding_window_inference(pose_array: np.ndarray, cls_model, reg_model,
                              body_weight_kg: float, window_size: int = T,
                              step: int = WINDOW_STEP) -> dict:
    """Run sliding window classification + regression on a full-video pose array.
    
    Returns dict with:
        window_starts, window_centers, predictions (class indices), 
        confidences, all_probs (class probs), 
        force_predictions (N), speed_predictions (m/s)
    """
    total_frames = pose_array.shape[0]
    
    if total_frames < window_size:
        raise ValueError(f"Video too short: {total_frames} frames, need {window_size}")
    
    window_starts = list(range(0, total_frames - window_size + 1, step))
    n_windows = len(window_starts)
    
    # Pre-compute all windows
    windows = np.zeros((n_windows, window_size, N_JOINTS, N_CHANNELS), dtype=np.float32)
    for i, start in enumerate(window_starts):
        windows[i] = pose_array[start:start + window_size]
    
    # Reshape: (n_windows, T, 9, 3) → (n_windows, 27, T)
    windows_flat = windows.reshape(n_windows, window_size, -1).transpose(0, 2, 1)
    tensor = torch.from_numpy(windows_flat).to(DEVICE)
    
    # Body weight: same for all windows
    body_weight_tensor = torch.full((n_windows,), body_weight_kg,
                                      dtype=torch.float32).to(DEVICE)
    
    # Run both models in batch
    with torch.no_grad():
        # Classification
        cls_logits = cls_model(tensor)
        cls_probs = F.softmax(cls_logits, dim=1).cpu().numpy()
        
        # Regression
        reg_pred_norm = reg_model(tensor, body_weight_tensor)
        # Un-normalize: pred_real = pred_norm * std + mean
        reg_pred = reg_pred_norm * target_std_tensor + target_mean_tensor
        reg_pred = reg_pred.cpu().numpy()
    
    predictions = cls_probs.argmax(axis=1)
    confidences = cls_probs.max(axis=1)
    window_centers = np.array(window_starts) + window_size // 2
    
    return {
        "window_starts": np.array(window_starts),
        "window_centers": window_centers,
        "predictions": predictions,
        "confidences": confidences,
        "all_probs": cls_probs,
        "force_predictions": reg_pred[:, 0],  # N
        "speed_predictions": reg_pred[:, 1],  # m/s
    }


# Run inference
pose_array = np.load(test_video["pose_path"])
print(f"Pose array shape: {pose_array.shape}")

results = sliding_window_inference(pose_array, cls_model, reg_model, body_weight_kg)
print(f"\nTotal windows: {len(results['predictions'])}")

# Class distribution
unique, counts = np.unique(results["predictions"], return_counts=True)
print(f"\nWindow-level class distribution:")
for u, c in zip(unique, counts):
    print(f"  {IDX_TO_CLASS[u]}: {c} ({100*c/len(results['predictions']):.1f}%)")

## Post-Processing: Merge Predictions into Punch Events

For each detected punch event, aggregate the regression predictions across
its windows (using the maximum, since we want peak values).

In [ ]:
def merge_predictions_to_events(results: dict,
                                 confidence_threshold: float = CONFIDENCE_THRESHOLD,
                                 min_punch_length: int = MIN_PUNCH_LENGTH,
                                 max_gap_to_merge: int = MAX_GAP_TO_MERGE) -> list[dict]:
    """Convert sliding window predictions into detected punch events with force/speed."""
    predictions = results["predictions"].copy()
    confidences = results["confidences"]
    window_starts = results["window_starts"]
    force_preds = results["force_predictions"]
    speed_preds = results["speed_predictions"]
    window_size = T
    
    no_punch_idx = CLASS_TO_IDX["no_punch"]
    
    # Step 1: low-confidence → no_punch
    low_conf_mask = confidences < confidence_threshold
    predictions[low_conf_mask] = no_punch_idx
    
    # Step 2: group consecutive same-class windows
    events = []
    if len(predictions) == 0:
        return events
    
    current_class = predictions[0]
    current_start_window = 0
    current_force_vals = [force_preds[0]]
    current_speed_vals = [speed_preds[0]]
    current_confs = [confidences[0]]
    
    for i in range(1, len(predictions)):
        if predictions[i] == current_class:
            current_force_vals.append(force_preds[i])
            current_speed_vals.append(speed_preds[i])
            current_confs.append(confidences[i])
        else:
            event_start = window_starts[current_start_window]
            event_end = window_starts[i - 1] + window_size - 1
            events.append({
                "start_frame": int(event_start),
                "end_frame": int(event_end),
                "class": IDX_TO_CLASS[current_class],
                "confidence": float(np.mean(current_confs)),
                "peak_force_N": float(np.max(current_force_vals)),
                "peak_speed_mps": float(np.max(current_speed_vals)),
            })
            current_class = predictions[i]
            current_start_window = i
            current_force_vals = [force_preds[i]]
            current_speed_vals = [speed_preds[i]]
            current_confs = [confidences[i]]
    
    # Last event
    event_start = window_starts[current_start_window]
    event_end = window_starts[-1] + window_size - 1
    events.append({
        "start_frame": int(event_start),
        "end_frame": int(event_end),
        "class": IDX_TO_CLASS[current_class],
        "confidence": float(np.mean(current_confs)),
        "peak_force_N": float(np.max(current_force_vals)),
        "peak_speed_mps": float(np.max(current_speed_vals)),
    })
    
    # Step 3: merge same-class events with small gaps
    merged = []
    for event in events:
        if merged and merged[-1]["class"] == event["class"] and \
           event["start_frame"] - merged[-1]["end_frame"] <= max_gap_to_merge:
            merged[-1]["end_frame"] = event["end_frame"]
            merged[-1]["confidence"] = (merged[-1]["confidence"] + event["confidence"]) / 2
            merged[-1]["peak_force_N"] = max(merged[-1]["peak_force_N"], event["peak_force_N"])
            merged[-1]["peak_speed_mps"] = max(merged[-1]["peak_speed_mps"], event["peak_speed_mps"])
        else:
            merged.append(event)
    
    # Step 4: filter short events (only for punch classes)
    final = [
        e for e in merged
        if e["class"] == "no_punch" or (e["end_frame"] - e["start_frame"] + 1) >= min_punch_length
    ]
    
    # Step 5: keep only punch events in output
    detected_punches = [e for e in final if e["class"] != "no_punch"]
    
    return detected_punches


detected_punches = merge_predictions_to_events(results)
print(f"Detected {len(detected_punches)} punches:")
print(f"\n{'#':>3}  {'Class':<10}  {'Frames':>13}  {'Force (N)':>10}  {'Speed (m/s)':>11}  {'Conf':>6}")
print("-" * 70)
for i, p in enumerate(detected_punches, 1):
    print(f"{i:>3}  {p['class']:<10}  {p['start_frame']:>5}-{p['end_frame']:<5}  "
          f"{p['peak_force_N']:>10.1f}  {p['peak_speed_mps']:>11.2f}  {p['confidence']:>6.3f}")

## Annotated Video Output

Generate an MP4 with predictions overlaid on each frame for presentation.
Shows class label, peak force, and peak speed for each detected punch.
"""

In [ ]:
import cv2
import sys

# Import rotation helpers from src/video.py (same logic as main.py)
sys.path.insert(0, str(PROJECT_ROOT))
from src.video import video_rotation

ANNOTATED_OUTPUT_DIR = PROJECT_ROOT / "data" / "annotated_videos"
ANNOTATED_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LABEL_DISPLAY_FRAMES = 30      # how long the label stays visible after midpoint
NO_PUNCH_GAP_FRAMES = 15       # gap before showing no_punch label


def find_source_video(video_stem: str) -> Path:
    """Find the source MP4/MOV for a given video stem."""
    raw_roots = [
        PROJECT_ROOT / "data" / "raw" / "with_hardware",
        PROJECT_ROOT / "data" / "raw" / "no_hardware",
    ]
    for raw_root in raw_roots:
        for ext in ("*.mp4", "*.mov", "*.MOV", "*.MP4", "*.avi"):
            for f in raw_root.rglob(ext):
                if f.stem == video_stem:
                    return f
    return None


def make_annotated_video(source_video: Path, output_path: Path,
                         detected_punches: list[dict]):
    """Render an annotated MP4 with predictions overlaid."""
    
    class_colors_bgr = {
        "cross":    (180, 119, 31),
        "hook":     (14, 127, 255),
        "jab":      (44, 160, 44),
        "uppercut": (40, 39, 214),
        "no_punch": (200, 200, 200),
    }
    
    # Get rotation metadata so output matches pose extraction orientation
    rotation = video_rotation(source_video)
    
    cap = cv2.VideoCapture(str(source_video))
    if not cap.isOpened():
        raise IOError(f"Cannot open video: {source_video}")
    
    fps = cap.get(cv2.CAP_PROP_FPS)
    raw_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    raw_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    # Adjust output dimensions based on rotation
    if rotation in (90, 270):
        width, height = raw_height, raw_width
    else:
        width, height = raw_width, raw_height
    
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(str(output_path), fourcc, fps, (width, height))
    
    # Build per-frame label tracking
    frame_to_border_class = ["no_punch"] * total_frames
    frame_to_label_class = ["no_punch"] * total_frames
    frame_to_label_conf = [0.0] * total_frames
    frame_to_label_force = [0.0] * total_frames
    frame_to_label_speed = [0.0] * total_frames
    
    # Mark border during entire punch
    for event in detected_punches:
        for f in range(event["start_frame"], min(event["end_frame"] + 1, total_frames)):
            frame_to_border_class[f] = event["class"]
    
    # Mark label starting at punch midpoint, for LABEL_DISPLAY_FRAMES frames
    for event in detected_punches:
        midpoint = (event["start_frame"] + event["end_frame"]) // 2
        label_end = min(midpoint + LABEL_DISPLAY_FRAMES, total_frames)
        for f in range(midpoint, label_end):
            frame_to_label_class[f] = event["class"]
            frame_to_label_conf[f] = event["confidence"]
            frame_to_label_force[f] = event["peak_force_N"]
            frame_to_label_speed[f] = event["peak_speed_mps"]
    
    # Insert no_punch labels during longer idle gaps
    last_label_end = -NO_PUNCH_GAP_FRAMES - 1
    for event in detected_punches:
        midpoint = (event["start_frame"] + event["end_frame"]) // 2
        event_label_end = min(midpoint + LABEL_DISPLAY_FRAMES, total_frames)
        gap_start = last_label_end + NO_PUNCH_GAP_FRAMES
        gap_end = midpoint
        if gap_end - gap_start > 0:
            for f in range(gap_start, gap_end):
                if frame_to_label_class[f] == "no_punch":
                    frame_to_label_class[f] = "no_punch"  # explicit no_punch label
        last_label_end = event_label_end
    
    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        # Apply rotation to match pose extraction orientation
        if rotation == 90:
            frame = cv2.rotate(frame, cv2.ROTATE_90_CLOCKWISE)
        elif rotation == 180:
            frame = cv2.rotate(frame, cv2.ROTATE_180)
        elif rotation == 270:
            frame = cv2.rotate(frame, cv2.ROTATE_90_COUNTERCLOCKWISE)
        
        if frame_idx < total_frames:
            border_class = frame_to_border_class[frame_idx]
            label_class = frame_to_label_class[frame_idx]
            label_conf = frame_to_label_conf[frame_idx]
            label_force = frame_to_label_force[frame_idx]
            label_speed = frame_to_label_speed[frame_idx]
        else:
            border_class = label_class = "no_punch"
            label_conf = label_force = label_speed = 0.0
        
        # Coloured border during active punch
        if border_class != "no_punch":
            color = class_colors_bgr[border_class]
            border = 10
            cv2.rectangle(frame, (0, 0), (width - 1, height - 1), color, border)
        
        # Text overlay box
        color = class_colors_bgr.get(label_class, (200, 200, 200))
        if label_class != "no_punch":
            # Punch detected — show class, force, speed
            cv2.rectangle(frame, (0, 0), (380, 110), (0, 0, 0), -1)
            cv2.putText(frame, f"{label_class.upper()}", (10, 35),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.0, color, 2, cv2.LINE_AA)
            cv2.putText(frame, f"Force: {label_force:.0f} N", (10, 65),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2, cv2.LINE_AA)
            cv2.putText(frame, f"Speed: {label_speed:.2f} m/s", (10, 90),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2, cv2.LINE_AA)
        else:
            cv2.rectangle(frame, (0, 0), (250, 50), (0, 0, 0), -1)
            cv2.putText(frame, "no_punch", (10, 35),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2, cv2.LINE_AA)
        
        # Frame counter
        cv2.putText(frame, f"f={frame_idx}", (10, height - 15),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)
        
        writer.write(frame)
        frame_idx += 1
    
    cap.release()
    writer.release()
    print(f"Annotated video saved: {output_path}")
    print(f"Total frames written: {frame_idx}")


# Generate annotated video
source_video = find_source_video(test_video["video_stem"])
if source_video is None:
    print(f"Could not find source video file for: {test_video['video_stem']}")
else:
    output_path = ANNOTATED_OUTPUT_DIR / f"{test_video['video_stem']}_regression_annotated.mp4"
    make_annotated_video(source_video, output_path, detected_punches)

## Summary

Final summary of detected punches with regression labels.
"""

In [ ]:
print("=" * 60)
print("INFERENCE SUMMARY")
print("=" * 60)
print(f"Regression model:     {REGRESSION_MODEL_PATH.name}")
print(f"Classification model: {CLASSIFICATION_MODEL_PATH.name}")
print(f"Test video:           {test_video['subject_id']} / {test_video['video_stem']}")
print(f"Subject body weight:  {body_weight_kg} kg")
print(f"Window size (T):      {T}")
print(f"Window step:          {WINDOW_STEP}")
print(f"Confidence threshold: {CONFIDENCE_THRESHOLD}")
print(f"\nDetected punches:     {len(detected_punches)}")

if detected_punches:
    forces = [p["peak_force_N"] for p in detected_punches]
    speeds = [p["peak_speed_mps"] for p in detected_punches]
    print(f"\nForce range:          {min(forces):.1f} - {max(forces):.1f} N (mean {np.mean(forces):.1f})")
    print(f"Speed range:          {min(speeds):.2f} - {max(speeds):.2f} m/s (mean {np.mean(speeds):.2f})")
    
    print(f"\nBy punch type:")
    from collections import defaultdict
    by_class = defaultdict(list)
    for p in detected_punches:
        by_class[p["class"]].append(p)
    for cls, punches in sorted(by_class.items()):
        forces = [p["peak_force_N"] for p in punches]
        speeds = [p["peak_speed_mps"] for p in punches]
        print(f"  {cls:<10s}: {len(punches)} punches, "
              f"force {np.mean(forces):.0f} N, speed {np.mean(speeds):.2f} m/s")

print("=" * 60)